# Healthcare Data Pipeline - Bronze, Silver, Gold

## Bronze Layer - Load & Explore Raw Data

In [0]:
%sql
SELECT * FROM patients_records
LIMIT 20;

Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281305978155,328,Urgent,2024-02-02,Paracetamol,Normal
LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327286577885,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096078842456,205,Emergency,2022-10-07,Aspirin,Normal
andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.78240987528,450,Elective,2020-12-18,Ibuprofen,Abnormal
adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317813937623,458,Urgent,2022-10-09,Penicillin,Abnormal
EMILY JOHNSOn,36,Male,A+,Asthma,2023-12-20,Taylor Newton,Nunez-Humphrey,UnitedHealthcare,48145.11095104189,389,Urgent,2023-12-24,Ibuprofen,Normal
edwArD EDWaRDs,21,Female,AB-,Diabetes,2020-11-03,Kelly Olson,Group Middleton,Medicare,19580.87234486093,389,Emergency,2020-11-15,Paracetamol,Inconclusive
CHrisTInA MARtinez,20,Female,A+,Cancer,2021-12-28,Suzanne Thomas,"Powell Robinson and Valdez,",Cigna,45820.46272159459,277,Emergency,2022-01-07,Paracetamol,Inconclusive
JASmINe aGuIlaR,82,Male,AB+,Asthma,2020-07-01,Daniel Ferguson,Sons Rich and,Cigna,50119.222791548505,316,Elective,2020-07-14,Aspirin,Abnormal
ChRISTopher BerG,58,Female,AB-,Cancer,2021-05-23,Heather Day,Padilla-Walker,UnitedHealthcare,19784.63106221073,249,Elective,2021-06-22,Paracetamol,Inconclusive


In [0]:
df = spark.table("patients_records")

In [0]:
df.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Blood Type: string (nullable = true)
 |-- Medical Condition: string (nullable = true)
 |-- Date of Admission: date (nullable = true)
 |-- Doctor: string (nullable = true)
 |-- Hospital: string (nullable = true)
 |-- Insurance Provider: string (nullable = true)
 |-- Billing Amount: double (nullable = true)
 |-- Room Number: long (nullable = true)
 |-- Admission Type: string (nullable = true)
 |-- Discharge Date: date (nullable = true)
 |-- Medication: string (nullable = true)
 |-- Test Results: string (nullable = true)



In [0]:
print(df.count())

55500


In [0]:
print(len(df.columns))
print(df.columns)

15
['Name', 'Age', 'Gender', 'Blood Type', 'Medical Condition', 'Date of Admission', 'Doctor', 'Hospital', 'Insurance Provider', 'Billing Amount', 'Room Number', 'Admission Type', 'Discharge Date', 'Medication', 'Test Results']


In [0]:
display(df.describe())

summary,Name,Age,Gender,Blood Type,Medical Condition,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Medication,Test Results
count,55500,55500,55500,55500,55500,55500,55500,55500,55500,55500,55500,55500,55500
mean,null,51.53945945945946,null,null,null,null,null,null,25539.31609721199,301.1348288288288,null,null,null
stddev,null,19.602453808514348,null,null,null,null,null,null,14211.454430864418,115.2430687009733,null,null,null
min,AARON DuncAn,13,Female,A+,Arthritis,Aaron Acevedo,Abbott Inc,Aetna,-2008.4921398591305,101,Elective,Aspirin,Abnormal
max,zachary WALl,89,Male,O-,Obesity,Zoe Wallace,"and Zuniga Thompson, Blake",UnitedHealthcare,52764.276736469175,500,Urgent,Penicillin,Normal


## Silver Layer - Rename Columns

In [0]:
silver_df=df

In [0]:
silver_df = (
    silver_df
    .withColumnRenamed("Name", "name")
    .withColumnRenamed("Age", "age")
    .withColumnRenamed("Gender", "gender")
    .withColumnRenamed("Blood Type", "blood_type")
    .withColumnRenamed("Medical Condition", "medical_condition")
    .withColumnRenamed("Date of Admission", "date_of_admission")
    .withColumnRenamed("Doctor", "doctor")
    .withColumnRenamed("Hospital", "hospital")
    .withColumnRenamed("Insurance Provider", "insurance_provider")
    .withColumnRenamed("Billing Amount", "billing_amount")
    .withColumnRenamed("Room Number", "room_number")
    .withColumnRenamed("Admission Type", "admission_type")
    .withColumnRenamed("Discharge Date", "discharge_date")
    .withColumnRenamed("Medication", "medication")
    .withColumnRenamed("Test Results", "test_results")
)

## Silver Layer - Data Quality Checks (Nulls & Duplicates)

In [0]:
from pyspark.sql.functions import col, when, count
display(
    silver_df.select(
        [count(when(col(c).isNull(), c)).alias(c) for c in silver_df.columns]
    )
)

name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
silver_df = silver_df.dropna()

In [0]:
print(silver_df.count())

55500


In [0]:
print("Before:",silver_df.count())
silver_df=silver_df.dropDuplicates()
print("After:",silver_df.count())

Before: 55500
After: 54966


## Silver Layer - Standardize Text Fields (Name / Doctor / Hospital Casing)

In [0]:
from pyspark.sql.functions import initcap
silver_df = silver_df.withColumn("name",initcap("name"))
silver_df = silver_df.withColumn("doctor",initcap("doctor"))
silver_df = silver_df.withColumn("hospital",initcap("hospital"))
display(silver_df.limit(10))

name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results
Bobby Jackson,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons And Miller,Blue Cross,18856.281305978155,328,Urgent,2024-02-02,Paracetamol,Normal
Leslie Terry,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327286577885,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
Danny Smith,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook Plc,Aetna,27955.096078842456,205,Emergency,2022-10-07,Aspirin,Normal
Andrew Watts,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers And Vang,",Medicare,37909.78240987528,450,Elective,2020-12-18,Ibuprofen,Abnormal
Adrienne Bell,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-white,Aetna,14238.317813937623,458,Urgent,2022-10-09,Penicillin,Abnormal
Emily Johnson,36,Male,A+,Asthma,2023-12-20,Taylor Newton,Nunez-humphrey,UnitedHealthcare,48145.11095104189,389,Urgent,2023-12-24,Ibuprofen,Normal
Edward Edwards,21,Female,AB-,Diabetes,2020-11-03,Kelly Olson,Group Middleton,Medicare,19580.87234486093,389,Emergency,2020-11-15,Paracetamol,Inconclusive
Christina Martinez,20,Female,A+,Cancer,2021-12-28,Suzanne Thomas,"Powell Robinson And Valdez,",Cigna,45820.46272159459,277,Emergency,2022-01-07,Paracetamol,Inconclusive
Jasmine Aguilar,82,Male,AB+,Asthma,2020-07-01,Daniel Ferguson,Sons Rich And,Cigna,50119.222791548505,316,Elective,2020-07-14,Aspirin,Abnormal
Christopher Berg,58,Female,AB-,Cancer,2021-05-23,Heather Day,Padilla-walker,UnitedHealthcare,19784.63106221073,249,Elective,2021-06-22,Paracetamol,Inconclusive


## Silver Layer - Feature Engineering (stay_days, billing_category, age_group)

In [0]:
from pyspark.sql.functions import datediff
silver_df = silver_df.withColumn("stay_days",
datediff("discharge_date","date_of_admission"))
display(silver_df.limit(10))

name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results,stay_days
Bobby Jackson,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons And Miller,Blue Cross,18856.281305978155,328,Urgent,2024-02-02,Paracetamol,Normal,2
Leslie Terry,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327286577885,265,Emergency,2019-08-26,Ibuprofen,Inconclusive,6
Danny Smith,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook Plc,Aetna,27955.096078842456,205,Emergency,2022-10-07,Aspirin,Normal,15
Andrew Watts,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers And Vang,",Medicare,37909.78240987528,450,Elective,2020-12-18,Ibuprofen,Abnormal,30
Adrienne Bell,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-white,Aetna,14238.317813937623,458,Urgent,2022-10-09,Penicillin,Abnormal,20
Emily Johnson,36,Male,A+,Asthma,2023-12-20,Taylor Newton,Nunez-humphrey,UnitedHealthcare,48145.11095104189,389,Urgent,2023-12-24,Ibuprofen,Normal,4
Edward Edwards,21,Female,AB-,Diabetes,2020-11-03,Kelly Olson,Group Middleton,Medicare,19580.87234486093,389,Emergency,2020-11-15,Paracetamol,Inconclusive,12
Christina Martinez,20,Female,A+,Cancer,2021-12-28,Suzanne Thomas,"Powell Robinson And Valdez,",Cigna,45820.46272159459,277,Emergency,2022-01-07,Paracetamol,Inconclusive,10
Jasmine Aguilar,82,Male,AB+,Asthma,2020-07-01,Daniel Ferguson,Sons Rich And,Cigna,50119.222791548505,316,Elective,2020-07-14,Aspirin,Abnormal,13
Christopher Berg,58,Female,AB-,Cancer,2021-05-23,Heather Day,Padilla-walker,UnitedHealthcare,19784.63106221073,249,Elective,2021-06-22,Paracetamol,Inconclusive,30


In [0]:
from pyspark.sql.functions import when,col
silver_df = silver_df.withColumn("billing_category",
when(col("billing_amount")<15000,"Low")
.when(col("billing_amount")<30000,"Medium")
.otherwise("High")
)
display(silver_df.limit(10))

name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results,stay_days,billing_category
Bobby Jackson,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons And Miller,Blue Cross,18856.281305978155,328,Urgent,2024-02-02,Paracetamol,Normal,2,Medium
Leslie Terry,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327286577885,265,Emergency,2019-08-26,Ibuprofen,Inconclusive,6,High
Danny Smith,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook Plc,Aetna,27955.096078842456,205,Emergency,2022-10-07,Aspirin,Normal,15,Medium
Andrew Watts,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers And Vang,",Medicare,37909.78240987528,450,Elective,2020-12-18,Ibuprofen,Abnormal,30,High
Adrienne Bell,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-white,Aetna,14238.317813937623,458,Urgent,2022-10-09,Penicillin,Abnormal,20,Low
Emily Johnson,36,Male,A+,Asthma,2023-12-20,Taylor Newton,Nunez-humphrey,UnitedHealthcare,48145.11095104189,389,Urgent,2023-12-24,Ibuprofen,Normal,4,High
Edward Edwards,21,Female,AB-,Diabetes,2020-11-03,Kelly Olson,Group Middleton,Medicare,19580.87234486093,389,Emergency,2020-11-15,Paracetamol,Inconclusive,12,Medium
Christina Martinez,20,Female,A+,Cancer,2021-12-28,Suzanne Thomas,"Powell Robinson And Valdez,",Cigna,45820.46272159459,277,Emergency,2022-01-07,Paracetamol,Inconclusive,10,High
Jasmine Aguilar,82,Male,AB+,Asthma,2020-07-01,Daniel Ferguson,Sons Rich And,Cigna,50119.222791548505,316,Elective,2020-07-14,Aspirin,Abnormal,13,High
Christopher Berg,58,Female,AB-,Cancer,2021-05-23,Heather Day,Padilla-walker,UnitedHealthcare,19784.63106221073,249,Elective,2021-06-22,Paracetamol,Inconclusive,30,Medium


In [0]:
silver_df = silver_df.withColumn("age_group",
when(col("age")<18,"Child")
.when(col("age")<60,"Adult")
.otherwise("Senior")
)
display(silver_df.limit(10))

name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results,stay_days,billing_category,age_group
Bobby Jackson,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons And Miller,Blue Cross,18856.281305978155,328,Urgent,2024-02-02,Paracetamol,Normal,2,Medium,Adult
Leslie Terry,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327286577885,265,Emergency,2019-08-26,Ibuprofen,Inconclusive,6,High,Senior
Danny Smith,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook Plc,Aetna,27955.096078842456,205,Emergency,2022-10-07,Aspirin,Normal,15,Medium,Senior
Andrew Watts,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers And Vang,",Medicare,37909.78240987528,450,Elective,2020-12-18,Ibuprofen,Abnormal,30,High,Adult
Adrienne Bell,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-white,Aetna,14238.317813937623,458,Urgent,2022-10-09,Penicillin,Abnormal,20,Low,Adult
Emily Johnson,36,Male,A+,Asthma,2023-12-20,Taylor Newton,Nunez-humphrey,UnitedHealthcare,48145.11095104189,389,Urgent,2023-12-24,Ibuprofen,Normal,4,High,Adult
Edward Edwards,21,Female,AB-,Diabetes,2020-11-03,Kelly Olson,Group Middleton,Medicare,19580.87234486093,389,Emergency,2020-11-15,Paracetamol,Inconclusive,12,Medium,Adult
Christina Martinez,20,Female,A+,Cancer,2021-12-28,Suzanne Thomas,"Powell Robinson And Valdez,",Cigna,45820.46272159459,277,Emergency,2022-01-07,Paracetamol,Inconclusive,10,High,Adult
Jasmine Aguilar,82,Male,AB+,Asthma,2020-07-01,Daniel Ferguson,Sons Rich And,Cigna,50119.222791548505,316,Elective,2020-07-14,Aspirin,Abnormal,13,High,Senior
Christopher Berg,58,Female,AB-,Cancer,2021-05-23,Heather Day,Padilla-walker,UnitedHealthcare,19784.63106221073,249,Elective,2021-06-22,Paracetamol,Inconclusive,30,Medium,Adult


## Silver Layer - Validate Categorical Values

In [0]:
silver_df.select("gender").distinct().show()
silver_df.select("blood_type").distinct().show()
silver_df.select("admission_type").distinct().show()
silver_df.select("test_results").distinct().show()

+------+
|gender|
+------+
|  Male|
|Female|
+------+

+----------+
|blood_type|
+----------+
|        B-|
|        A+|
|        A-|
|        O+|
|       AB+|
|       AB-|
|        B+|
|        O-|
+----------+

+--------------+
|admission_type|
+--------------+
|        Urgent|
|     Emergency|
|      Elective|
+--------------+

+------------+
|test_results|
+------------+
|      Normal|
|Inconclusive|
|    Abnormal|
+------------+



## Silver Layer - Generate Patient ID & Save Silver Table

In [0]:
from pyspark.sql.functions import monotonically_increasing_id
silver_df = silver_df.withColumn(
    "patient_id",
    monotonically_increasing_id()
)

In [0]:
silver_df.write.format("delta").mode("overwrite").saveAsTable("silver_patients")

## Silver Layer - SCD Type 2 History Table (Initial Load)

In [0]:
from pyspark.sql.functions import current_date, lit

silver_scd = silver_df \
    .withColumn("start_date", current_date()) \
    .withColumn("end_date", lit(None).cast("date")) \
    .withColumn("is_current", lit(True))

silver_scd.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_patients_history")

In [0]:
%sql
SELECT * FROM silver_patients_history LIMIT 10;

name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results,stay_days,billing_category,age_group,patient_id,start_date,end_date,is_current
Brooke Brady,44,Female,AB+,Cancer,2021-10-08,Roberta Stewart,Morris-arellano,UnitedHealthcare,40701.599227308754,182,Urgent,2021-10-13,Paracetamol,Normal,5,High,Adult,0,2026-07-19,null,true
Carol Patterson,29,Female,O+,Cancer,2022-10-24,Jamie Baker,"Turner Cook, Banks And",Blue Cross,19328.118579079928,231,Urgent,2022-11-07,Ibuprofen,Normal,14,Medium,Adult,1,2026-07-19,null,true
Chad Moreno,67,Male,AB+,Hypertension,2020-08-26,Connie Boyd,Inc Skinner,Aetna,46814.011195111656,134,Urgent,2020-08-27,Penicillin,Abnormal,1,High,Senior,2,2026-07-19,null,true
Michael Martin,84,Male,A+,Asthma,2022-09-06,John Summers,Sons Horn And,Cigna,23684.52547274483,162,Urgent,2022-09-27,Ibuprofen,Inconclusive,21,Medium,Senior,3,2026-07-19,null,true
William Morton,21,Male,A+,Diabetes,2023-06-25,Christina Hammond,Thompson-walker,UnitedHealthcare,3125.7364766012533,442,Urgent,2023-07-11,Ibuprofen,Normal,16,Low,Adult,4,2026-07-19,null,true
Christina Schmitt,53,Male,B+,Cancer,2023-06-08,Samuel Robles,Cruz Ltd,UnitedHealthcare,27360.46189055129,231,Elective,2023-07-05,Ibuprofen,Abnormal,27,Medium,Adult,5,2026-07-19,null,true
Jeffrey Turner,85,Female,O+,Obesity,2020-05-31,Matthew Carter,"Kim Rosario, And Hammond",Aetna,39957.94062222976,211,Urgent,2020-06-01,Ibuprofen,Normal,1,High,Senior,6,2026-07-19,null,true
Robyn Miranda,30,Female,A-,Diabetes,2020-10-11,Elizabeth Frank,"Walker And Gardner Fernandez,",Medicare,42792.24128217486,468,Elective,2020-10-21,Ibuprofen,Abnormal,10,High,Adult,7,2026-07-19,null,true
Robert Walsh,20,Female,B+,Cancer,2020-11-22,Sabrina Rogers,"And Anderson Smith Sanchez,",Cigna,40598.42257087868,113,Elective,2020-12-17,Paracetamol,Normal,25,High,Adult,8,2026-07-19,null,true
Mark Lawrence,19,Female,O-,Diabetes,2022-02-08,Melissa Terry,Mcneil-blake,Blue Cross,10348.818521438636,271,Emergency,2022-02-17,Ibuprofen,Inconclusive,9,Low,Adult,9,2026-07-19,null,true


## SCD Type 2 - Simulate an Incoming Change

In [0]:
display(silver_df.select("patient_id","name","billing_amount").limit(10))

patient_id,name,billing_amount
0,Bobby Jackson,18856.281305978155
1,Leslie Terry,33643.327286577885
2,Danny Smith,27955.096078842456
3,Andrew Watts,37909.78240987528
4,Adrienne Bell,14238.317813937623
5,Emily Johnson,48145.11095104189
6,Edward Edwards,19580.87234486093
7,Christina Martinez,45820.46272159459
8,Jasmine Aguilar,50119.222791548505
9,Christopher Berg,19784.63106221073


In [0]:
from pyspark.sql.functions import when, col

new_df = silver_df.withColumn(
    "billing_amount",
    when(col("patient_id") == 0, 25000)
    .otherwise(col("billing_amount"))
)
new_df.createOrReplaceTempView("new_patients")

## SCD Type 2 - MERGE to Track Changes

In [0]:
%sql
MERGE INTO silver_patients_history AS target
USING new_patients AS source
ON target.patient_id = source.patient_id
AND target.is_current = true
WHEN MATCHED
AND (
    target.billing_amount <> source.billing_amount
    OR target.doctor <> source.doctor
)
THEN
UPDATE SET
    target.end_date = current_date(),
    target.is_current = false;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,1,0,0


In [0]:
%sql
INSERT INTO silver_patients_history
SELECT
    name,
    age,
    gender,
    blood_type,
    medical_condition,
    date_of_admission,
    doctor,
    hospital,
    insurance_provider,
    billing_amount,
    room_number,
    admission_type,
    discharge_date,
    medication,
    test_results,
    stay_days,
    billing_category,
    age_group,
    patient_id,
    current_date() AS start_date,
    NULL AS end_date,
    true AS is_current
FROM new_patients
WHERE patient_id = 0
AND billing_amount = 25000;

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
SELECT patient_id,
       name,
       billing_amount,
       start_date,
       end_date,
       is_current
FROM silver_patients_history
WHERE patient_id = 0;

patient_id,name,billing_amount,start_date,end_date,is_current
0,Brooke Brady,25000.0,2026-07-19,null,true
0,Brooke Brady,40701.599227308754,2026-07-19,2026-07-19,false


# Gold Layer - Business Insight Tables

In [0]:
%sql
CREATE OR REPLACE TABLE gold_patient_per_hospital AS
SELECT
hospital,
COUNT(*) AS total_patients
FROM silver_patients
GROUP BY hospital;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_avg_billing AS
SELECT
hospital,
ROUND(AVG(billing_amount),2) AvgBilling
FROM silver_patients
GROUP BY hospital;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_insurance_analysis AS
SELECT
`insurance_provider`,
COUNT(*) total_patients,
ROUND(AVG(billing_amount),2) AvgBill
FROM silver_patients
GROUP BY `insurance_provider`;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_gender_distribution AS
SELECT
gender,
COUNT(*) total
FROM silver_patients
GROUP BY gender;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_admission_type AS
SELECT
`admission_type`,
COUNT(*) total
FROM silver_patients
GROUP BY `admission_type`;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_hospital_ranking AS
SELECT
hospital,
SUM(billing_amount) AS revenue,
RANK() OVER(ORDER BY SUM(billing_amount) DESC) AS hospital_rank
FROM silver_patients
GROUP BY hospital;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_doctor_performance AS
SELECT
doctor,
COUNT(*) AS total_patients
FROM silver_patients
GROUP BY doctor
ORDER BY total_patients DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_top_diseases AS
SELECT
`medical_condition`,
COUNT(*) total_patients
FROM silver_patients
GROUP BY `medical_condition`
ORDER BY total_patients DESC;

num_affected_rows,num_inserted_rows


## Gold Layer - View Results

In [0]:
%sql
SELECT * FROM gold_patient_per_hospital LIMIT 20;

hospital,total_patients
Fox-singleton,1
"Walsh Alexander, And Dennis",1
Alvarado Plc,3
Murphy Plc,1
Collins-gilbert,1
Johnson-nichols,2
Serrano-griffin,1
Johnson-banks,1
Llc Jimenez,3
"Martin Taylor Johnson, And",1


In [0]:
%sql
SELECT * FROM gold_avg_billing LIMIT 10;

hospital,AvgBilling
Fox-singleton,47624.25
"Walsh Alexander, And Dennis",19993.54
Alvarado Plc,30875.49
Murphy Plc,29063.27
Collins-gilbert,27081.57
Johnson-nichols,14558.19
Serrano-griffin,50173.25
Johnson-banks,17933.84
Llc Jimenez,23215.52
"Martin Taylor Johnson, And",8196.57


In [0]:
%sql
SELECT * FROM gold_insurance_analysis;

insurance_provider,total_patients,AvgBill
Medicare,11039,25628.32
Blue Cross,10952,25603.46
Cigna,11139,25526.0
Aetna,10822,25549.69
UnitedHealthcare,11014,25414.51


In [0]:
%sql
SELECT * FROM gold_gender_distribution;

gender,total
Male,27496
Female,27470


In [0]:
%sql
SELECT * FROM gold_admission_type;

admission_type,total
Emergency,18102
Elective,18473
Urgent,18391


In [0]:
%sql
SELECT * FROM gold_hospital_ranking LIMIT 10;

hospital,revenue,hospital_rank
Johnson Plc,1081477.3119877572,1
Llc Smith,1030189.8722479499,2
Smith Plc,1029424.4491163141,3
Ltd Smith,1003365.5277071944,4
Smith Ltd,970035.8657058083,5
Johnson Inc,934310.728058776,6
Group Smith,902975.7874592737,7
Inc Brown,877961.3147981078,8
Llc Johnson,816438.3536878258,9
Smith Group,806631.2907331102,10


In [0]:
%sql
SELECT * FROM gold_doctor_performance LIMIT 10;

doctor,total_patients
Michael Smith,27
John Smith,22
Robert Smith,21
Michael Johnson,20
James Smith,20
Robert Johnson,19
David Smith,19
Michael Williams,18
Christopher Smith,17
John Johnson,17


In [0]:
%sql
SELECT * FROM gold_top_diseases;

medical_condition,total_patients
Arthritis,9218
Diabetes,9216
Hypertension,9151
Obesity,9146
Cancer,9140
Asthma,9095
